# Hello LangGraph
이곳에서 LangGraph 에이전트 테스트를 진행할 수 있습니다.

In [24]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import  TypedDict
from typing import Annotated
import operator
from datetime import datetime
from typing import Literal
from typing import Union
from langchain.chat_models import init_chat_model
from langchain_core.messages import AnyMessage
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
from langgraph.types import interrupt, Command

llm = init_chat_model("openai:gpt-4o")

conn = sqlite3.connect("memory.db", check_same_thread=False,)

In [25]:
class State(MessagesState):
    custom_stuff : str


graph_builder = StateGraph(State)


In [26]:
# @tool
# def get_human_feedback(poem : str):
#     """Asks the user for feedback on a poem.
#         Use this before returning the fianl response
#     """
#     feedback = interrupt(f"Here is the poem, tell me what you think \n{poem}")
#     return feedback

# llm_with_tools = llm.bind_tools([get_human_feedback])

# def chatbot(state : State):
#     response = llm_with_tools.invoke(f"""
#     You are an expert in making poems.

#     Use the `get_human_feedback` tool to ask the user for feedback on the poem you generated.
    
#     Only after you receive positive feedback you can return the final poem. 
                                     
#     ALWATYS ASK FOR FEEDBACK FIRST.
                                     
#     Here is the conversation history:
                                     
#     {state["messages"]}
                                     
#     """)

#     return {
#         "messages" : [response]
#     }

# tool_node = ToolNode(
#     tools=[
#         get_human_feedback
#     ]
# )

def chatbot(state : State):
    response = llm.invoke(state['messages'])

    return {
        "messages" : [response],
    }

In [27]:
graph_builder.add_node("chatbot", chatbot)
# graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
# graph_builder.add_conditional_edges("chatbot",tools_condition)
# graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge("chatbot", END)

graph = graph_builder.compile(
    checkpointer=SqliteSaver(conn)
)

config = {
    "configurable" : {
        "thread_id" : "12"
    },
}

result = graph.invoke(
    {
        "messages" : [
            {"role":"user", "content": "현재 나는 대한민국 서울에 살고 있어"}
        ]
    },
    config=config
)

for message in result["messages"]:
    message.pretty_print()


================================ Human Message =================================

안녕 내 이름은 김정태여
================================== Ai Message ==================================

안녕하세요, 김정태님! 만나서 반갑습니다. 오늘 어떻게 도와드릴까요?
================================ Human Message =================================

현재 나는 대한민국 서울에 살고 있어
================================== Ai Message ==================================

서울에 살고 계시는군요! 서울은 정말 다양한 문화와 맛있는 음식이 가득한 곳이죠. 특별히 즐겨 찾는 장소나 추천하고 싶으신 곳이 있으신가요?


In [38]:
state_history = graph.get_state_history(config)

In [39]:
for state_snapshot in list(state_history):
    print(state_snapshot.next)
    print(state_snapshot.values["messages"])
    print("-----------------")

()
[HumanMessage(content='안녕 내 이름은 김정태여', additional_kwargs={}, response_metadata={}, id='fe46ba92-852e-43dd-8dff-82be09895c31'), AIMessage(content='안녕하세요, 김정태님! 만나서 반갑습니다. 오늘 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 16, 'total_tokens': 38, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_c8b70290c4', 'id': 'chatcmpl-DRIkcnJrkGh4T2INzxcsdUAzlOwar', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d5e0b-7f58-71f1-bcb3-cd6aaf52a375-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 22, 'total_tokens': 38, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details':

In [41]:
state_history = graph.get_state_history(config)

to_fork = list(state_history)[-5]

to_fork.config

{'configurable': {'thread_id': '12',
  'checkpoint_ns': '',
  'checkpoint_id': '1f130fbe-bc71-602c-8003-62339cef3654'}}

In [42]:
from langchain_core.messages import HumanMessage

# b4eb22df-0331-41af-8254-8f4c7366b47f

graph.update_state(
    to_fork.config,
    {
        "messages" : [
            HumanMessage(
                content="현재 나는 대한민국 성남에 살고 있어",
                id="b4eb22df-0331-41af-8254-8f4c7366b47f"
            )
        ]
    }
)

{'configurable': {'thread_id': '12',
  'checkpoint_ns': '',
  'checkpoint_id': '1f130ff0-407b-6c1a-8004-c66ae989dc36'}}

In [43]:
forked_state = graph.get_state_history(
    {
        'configurable': {
            'thread_id': '12',
            'checkpoint_ns': '',
            'checkpoint_id': '1f130ff0-407b-6c1a-8004-c66ae989dc36'
        }
    }
)

list(forked_state)

[StateSnapshot(values={'messages': [HumanMessage(content='안녕 내 이름은 김정태여', additional_kwargs={}, response_metadata={}, id='fe46ba92-852e-43dd-8dff-82be09895c31'), AIMessage(content='안녕하세요, 김정태님! 만나서 반갑습니다. 오늘 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 16, 'total_tokens': 38, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_c8b70290c4', 'id': 'chatcmpl-DRIkcnJrkGh4T2INzxcsdUAzlOwar', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d5e0b-7f58-71f1-bcb3-cd6aaf52a375-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 22, 'total_tokens': 38, 'input_token_details': {'audio': 0, 'cache_re

In [44]:
result = graph.invoke(
    None,
    {
        'configurable': {
            'thread_id': '12',
            'checkpoint_ns': '',
            'checkpoint_id': '1f130ff0-407b-6c1a-8004-c66ae989dc36'
        }
    }
)

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

안녕 내 이름은 김정태여
================================== Ai Message ==================================

안녕하세요, 김정태님! 만나서 반갑습니다. 오늘 어떻게 도와드릴까요?
================================ Human Message =================================

현재 나는 대한민국 성남에 살고 있어
================================== Ai Message ==================================

성남에 살고 계시는군요! 성남은 다양한 문화와 편의시설이 잘 갖춰진 곳으로 알려져 있죠. 거기서 생활하시는 데에 대해 더 알고 싶으시면 말씀해 주세요. 도움이 필요하시거나 궁금한 점이 있다면 언제든지 말씀해 주세요!
